In [ ]:
#!/usr/bin/env python
# coding: utf-8

In[1]:

Ask the user to input the name format

In [ ]:
name_format = "large_run11"
print("Run:", name_format)

Function to generate file names based on the input format

In [ ]:
def generate_filename(base_name, extension):
    return f"{name_format}_{base_name}.{extension}"

In[3]:

In [ ]:
import scipy.io
import os
import numpy as np
from scipy.interpolate import interp1d

In [ ]:
print("Loading data...", flush=True)

Define the paths to the .mat files

In [ ]:
b_file_path = os.path.join('MATLAB_DATA', 'B.mat')
e_file_path = os.path.join('MATLAB_DATA', 'EDC.mat')
x_file_path = os.path.join('MATLAB_DATA', 'X.mat')
i_file_path = os.path.join('MATLAB_DATA', 'I.mat')

Load the .mat files

In [ ]:
b_contents = scipy.io.loadmat(b_file_path)
e_contents = scipy.io.loadmat(e_file_path)
x_contents = scipy.io.loadmat(x_file_path)
i_contents = scipy.io.loadmat(i_file_path)

Access the variables from the loaded .mat files

In [ ]:
B_MAG = b_contents['results1']
EDC_MAG = e_contents['results2']
X_data_raw = x_contents['results3']
I_data_raw = i_contents['results4']

Scale the X values by 10^10 (assuming you still need this step)

In [ ]:
for i in range(len(X_data_raw)):
    X_data_raw[i][0][0][0] = X_data_raw[i][0][0][0] * 10**10

Define a fixed length for resampling

In [ ]:
fixed_length = 1200  # Adjust based on the average length of your data

Initialize lists to hold the resampled data

In [ ]:
X_data = []
I_data = []
B_MAG_data = []
EDC_MAG_data = []
# Function to resample and smooth data
def resample_and_smooth(X, I, new_length):
    # Define new X as evenly spaced values between 6562.5 and 6563.1
    X_new = np.linspace(6562.3, 6563.3, num=new_length)
    
    # Interpolate the I values over the new X values
    f = interp1d(X, I, kind='cubic', fill_value="extrapolate")  # 'cubic' provides smoothing
    I_new = f(X_new)
    
    return X_new, I_new

Iterate through the data and resample

In [ ]:
for i in range(len(X_data_raw)):
    X = X_data_raw[i][0][0][0]
    I = I_data_raw[i][0][0][0]
    B = B_MAG[i][0][0][0]
    EDC = EDC_MAG[i][0][0][0]
    # Resample X and smooth I correspondingly
    X_resampled, I_resampled = resample_and_smooth(X, I, fixed_length)
    
    # Store the resampled data in lists
    X_data.append(X_resampled)
    I_data.append(I_resampled)
    B_MAG_data.append(B)
    EDC_MAG_data.append(EDC)

Convert lists to numpy arrays

In [ ]:
X_data = np.array(X_data)
I_data = np.array(I_data)
B_MAG_data = np.array(B_MAG_data)
EDC_MAG_data = np.array(EDC_MAG_data)

In [ ]:
print("Data loaded and processed.", flush=True)

In[4]:

In [ ]:
import numpy as np

Assuming I_data is your input numpy array<br>
Step 1: Calculate the mean of each column (axis=0 means column-wise operation)

In [ ]:
means = np.mean(I_data, axis=0)

Step 2: Calculate the standard deviation of each column

In [ ]:
stds = np.std(I_data, axis=0)

Step 2: Initialize indices for the columns to keep

In [ ]:
start_index = 0
end_index = I_data.shape[1] - 1

Step 3: Find the first column with std larger than 0.01 from the start

In [ ]:
for i in range(I_data.shape[1]):
    if stds[i] >= 0.01:
        start_index = i
        break

Step 4: Find the first column with std larger than 0.01 from the end

In [ ]:
for i in range(I_data.shape[1] - 1, -1, -1):
    if stds[i] >= 0.01:
        end_index = i
        break

Step 5: Slice the I_data, B_MAG_data, and EDC_MAG_data based on the start and end indices

In [ ]:
I_data_filtered = I_data[:, start_index:end_index + 1]
X_resampled_filtered = X_resampled[start_index:end_index + 1]
means_filtered = means[start_index:end_index + 1]
stds_filtered = stds[start_index:end_index + 1]
print(f"Data filtered from column {start_index} to {end_index}.")

Step 3: Normalize the array (I_data - mean) / std

In [ ]:
normalized_I = (I_data_filtered - means_filtered) / stds_filtered

Now normalized_data contains the result of (I - mean) / std for each column.

In[5]:

In [ ]:
import torch
import sbi
from sbi import utils as sbi_utils
from sbi.inference import SNPE

In[5]:

Concatenate B and E to form the input data

In [ ]:
input_data = np.hstack((B_MAG_data, EDC_MAG_data))

Stack X and I to form the target data<br>
target_data = np.hstack((X_data, normalized_I))

In [ ]:
target_data = normalized_I

Convert lists to numpy arrays (if not already done)

In [ ]:
input_data = np.array(input_data)
target_data = np.array(target_data)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

Example: Normalize data

In [ ]:
scaler = MinMaxScaler(feature_range=(0, 1))
input_data= scaler.fit_transform(input_data)
# target_data= scaler.fit_transform(target_data)

In [ ]:
input_data_tensor = torch.tensor(input_data, dtype=torch.float32)
target_data_tensor = torch.tensor(target_data, dtype=torch.float32)

In[6]:

In [ ]:
from sklearn.model_selection import train_test_split

Split the data into training and testing sets

In [ ]:
input_data_train, input_data_test, target_data_train, target_data_test = train_test_split(
    input_data, target_data, test_size=0.2, random_state=42
)

Convert these splits to PyTorch tensors

In [ ]:
input_data_train_tensor = torch.tensor(input_data_train, dtype=torch.float32)  # Shape: (train_samples, 2400)
input_data_test_tensor = torch.tensor(input_data_test, dtype=torch.float32)  # Shape: (test_samples, 2400)
target_data_train_tensor = torch.tensor(target_data_train, dtype=torch.float32)  # Shape: (train_samples, 6)
target_data_test_tensor = torch.tensor(target_data_test, dtype=torch.float32)  # Shape: (test_samples, 6)

In [ ]:
print("Data organized.", flush=True)

In[7]:

Define prior bounds for your data. Adjust according to your specific problem.

In [ ]:
prior_min = torch.tensor([input_data_tensor.min().item()] * input_data_tensor.shape[1])
prior_max = torch.tensor([input_data_tensor.max().item()] * input_data_tensor.shape[1])

prior = sbi_utils.BoxUniform(low=prior_min, high=prior_max)

Widening the prior range slightly

In [ ]:
margin = 0.1 * (prior_max - prior_min)
prior = sbi_utils.BoxUniform(low=prior_min - margin, high=prior_max + margin)

In[8]:

Train the model

In [ ]:
print("Start training.", flush=True)

In [ ]:
from sbi.inference import SNPE, prepare_for_sbi
from sbi.utils.get_nn_models import posterior_nn

Initialize the SNPE with a custom neural network architecture

In [ ]:
neural_net = posterior_nn(model='nsf',  # Consider 'maf' (Masked Autoregressive Flow) or 'nsf' (Neural Spline Flow)
                          hidden_features=64,  # Increase the number of hidden features
                          num_transforms=10)  # Increase the number of transformations

Initialize SNPE with the prior and the custom neural network

In [ ]:
inference = SNPE(prior=prior, density_estimator=neural_net)

Set the training steps during the training process (not during `train()`)

In [ ]:
inference.append_simulations(input_data_train_tensor, target_data_train_tensor)
density_estimator = inference.train(max_num_epochs=200, learning_rate=0.001, training_batch_size=64)  # Increase epochs for potentially better results

Build posterior

In [ ]:
posterior = inference.build_posterior(density_estimator)
# Build the posterior using NUTS for sampling
# posterior = inference.build_posterior(density_estimator, sample_with='mcmc')

In [ ]:
print("Training done.", flush=True)

Save the model

In [ ]:
import torch

Define the file paths for saving

In [ ]:
posterior_path = os.path.join("/home/botingl/machine learning", generate_filename("posterior", "pt"))
density_estimator_path = os.path.join("/home/botingl/machine learning", generate_filename("density_estimator", "pt"))

Save the posterior (which includes the trained model)

In [ ]:
torch.save(posterior, posterior_path)

Save the density estimator state dictionary

In [ ]:
torch.save(density_estimator.state_dict(), density_estimator_path)

In [ ]:
print("Model saved successfully.")

In[ ]:

Load saved model

In [ ]:
import torch
from sbi.inference import SNPE
from sbi.utils.get_nn_models import posterior_nn

Define the file paths for loading

In [ ]:
posterior_path = os.path.join("/home/botingl/machine learning", generate_filename("posterior", "pt"))

Load the posterior

In [ ]:
posterior = torch.load(posterior_path)

In [ ]:
print("Model loaded successfully.")

In[ ]:

In [ ]:
print("Start to evaluate the training set.", flush=True)
# Generate predictions for all test sets
import sys
import numpy as np
from tqdm import tqdm

Limit the test set to only 10 groups for faster evaluation

In [ ]:
train_subset_indices = np.random.choice(len(input_data_train_tensor), size=1000, replace=False)
train_target_data_subset = target_data_train_tensor[train_subset_indices]
train_input_data_subset = input_data_train_tensor[train_subset_indices]

Generate predictions for the test set

In [ ]:
predictions = []
for i in tqdm(range(len(train_target_data_subset)), desc="Processing samples", leave=True, file=sys.stdout):
    test_input = train_target_data_subset[i]  # This is the X and I for this test sample
    predicted_posterior = posterior.sample((1000,), x=test_input, show_progress_bars=False)  # Disable internal progress bars
    
    # Extract mean prediction for B and E
    predicted_mean = predicted_posterior.mean(dim=0)
    predictions.append(predicted_mean)
    
    sys.stdout.flush()  # Manually flush output

Convert predictions to numpy array

In [ ]:
predictions = torch.stack(predictions).detach().numpy()

In [ ]:
print("Evaluation done for the training set.", flush=True)

In[ ]:

In [ ]:
output_dir = "/home/botingl/machine learning"
# Convert the true B and E values to numpy array
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
true_values = train_input_data_subset.numpy()

In [ ]:
def normalized_root_mean_squared_error(y_true, y_pred):
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return rmse / (y_true.max() - y_true.min())

Calculate evaluation metrics

In [ ]:
mae = mean_absolute_error(true_values, predictions)
rmse = np.sqrt(mean_squared_error(true_values, predictions))
r2 = r2_score(true_values, predictions)
nrmse = normalized_root_mean_squared_error(true_values, predictions)

In [ ]:
print(f'NRMSE: {nrmse:.2f}')
print(f'Mean Absolute Error (MAE): {mae}')
print(f'Root Mean Squared Error (RMSE): {rmse}')
print(f'R^2 Score: {r2}')

Save evaluation metrics to a text file

In [ ]:
metrics_file = os.path.join(output_dir, generate_filename("evaluation_metrics_train", "txt"))
with open(metrics_file, "w") as f:
    f.write(f"Mean Absolute Error (MAE): {mae}\n")
    f.write(f"Root Mean Squared Error (RMSE): {rmse}\n")
    f.write(f"R^2 Score: {r2}\n")
    f.write(f"Normalized RMSE (NRMSE): {nrmse}\n")

Optionally, you can visualize the results

In [ ]:
import matplotlib.pyplot as plt

Plot the true vs. predicted B and E values

In [ ]:
plt.figure(figsize=(15, 10))

Plotting for B components

In [ ]:
for i in range(3):
    plt.subplot(2, 3, i+1)
    plt.scatter(true_values[:, i], predictions[:, i], alpha=0.5)
    plt.xlabel(f'True B[{i+1}]')
    plt.ylabel(f'Predicted B[{i+1}]')
    plt.title(f'True vs. Predicted B[{i+1}]')

Plotting for E components

In [ ]:
for i in range(3):
    plt.subplot(2, 3, i+4)
    plt.scatter(true_values[:, i+3], predictions[:, i+3], alpha=0.5)
    plt.xlabel(f'True E[{i+1}]')
    plt.ylabel(f'Predicted E[{i+1}]')
    plt.title(f'True vs. Predicted E[{i+1}]')

In [ ]:
plt.tight_layout()

In [ ]:
figure_file = os.path.join(output_dir, generate_filename("true_vs_predictions_train", "png"))
plt.savefig(figure_file, dpi=300, facecolor='white')
plt.show()
plt.close()

In[ ]:

In [ ]:
print("Start to evaluate the test set.", flush=True)
# Generate predictions for all test sets
import sys
import numpy as np
from tqdm import tqdm

Generate predictions for the test set

In [ ]:
predictions = []
for i in tqdm(range(len(target_data_test)), desc="Processing samples", leave=True, file=sys.stdout):
    test_input = target_data_test[i]  # This is the X and I for this test sample
    predicted_posterior = posterior.sample((1000,), x=test_input, show_progress_bars=False)  # Disable internal progress bars
    
    # Extract mean prediction for B and E
    predicted_mean = predicted_posterior.mean(dim=0)
    predictions.append(predicted_mean)
    
    sys.stdout.flush()  # Manually flush output

Convert predictions to numpy array

In [ ]:
predictions = torch.stack(predictions).detach().numpy()

In [ ]:
print("Evaluation done for the test set.", flush=True)

In[ ]:

Evaluate the model using all test sets

In [ ]:
output_dir = "/home/botingl/machine learning"
# Convert the true B and E values to numpy array
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
true_values = input_data_test

In [ ]:
def normalized_root_mean_squared_error(y_true, y_pred):
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    return rmse / (y_true.max() - y_true.min())

Calculate evaluation metrics

In [ ]:
mae = mean_absolute_error(true_values, predictions)
rmse = np.sqrt(mean_squared_error(true_values, predictions))
r2 = r2_score(true_values, predictions)
nrmse = normalized_root_mean_squared_error(input_data_test, predictions)

In [ ]:
print(f'NRMSE: {nrmse:.2f}')
print(f'Mean Absolute Error (MAE): {mae}')
print(f'Root Mean Squared Error (RMSE): {rmse}')
print(f'R^2 Score: {r2}')

Save evaluation metrics to a text file

In [ ]:
metrics_file = os.path.join(output_dir, generate_filename("evaluation_metrics_test", "txt"))
with open(metrics_file, "w") as f:
    f.write(f"Mean Absolute Error (MAE): {mae}\n")
    f.write(f"Root Mean Squared Error (RMSE): {rmse}\n")
    f.write(f"R^2 Score: {r2}\n")
    f.write(f"Normalized RMSE (NRMSE): {nrmse}\n")

Optionally, you can visualize the results

In [ ]:
import matplotlib.pyplot as plt

Plot the true vs. predicted B and E values

In [ ]:
plt.figure(figsize=(15, 10))

Plotting for B components

In [ ]:
for i in range(3):
    plt.subplot(2, 3, i+1)
    plt.scatter(true_values[:, i], predictions[:, i], alpha=0.5)
    plt.xlabel(f'True B[{i+1}]')
    plt.ylabel(f'Predicted B[{i+1}]')
    plt.title(f'True vs. Predicted B[{i+1}]')

Plotting for E components

In [ ]:
for i in range(3):
    plt.subplot(2, 3, i+4)
    plt.scatter(true_values[:, i+3], predictions[:, i+3], alpha=0.5)
    plt.xlabel(f'True E[{i+1}]')
    plt.ylabel(f'Predicted E[{i+1}]')
    plt.title(f'True vs. Predicted E[{i+1}]')

In [ ]:
plt.tight_layout()

In [ ]:
figure_file = os.path.join(output_dir, generate_filename("true_vs_predictions_test", "png"))
plt.savefig(figure_file, dpi=300, facecolor='white')
plt.show()
plt.close()